# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, referencing all dataset elements by their `@id`s as defined in the Croissant schema.

### Dataset Source
The dataset Croissant schema is published at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the Croissant schema.

In [ ]:
# List all record sets by their @id and get summary of their fields and columns
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields (@id): {field_ids}")
    # If columns exist (for tabular data)
    column_ids = []
    for field in rs.fields:
        if hasattr(field, 'columns') and field.columns:
            column_ids.extend([column.id for column in field.columns])
    if column_ids:
        print(f"  Columns (@id): {column_ids}")
    print('-'*50)

# Save record set IDs for later
record_set_ids = [rs.id for rs in record_sets]

# Show an example of a record from the first record set (if any)
if record_set_ids:
    example_records = list(dataset.records(record_set=record_set_ids[0]))
    if example_records:
        print(f"\nSample record from record set '{record_sets[0].name}':\n{example_records[0]}")
    else:
        print(f"No records available in record set '{record_sets[0].name}'")
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis using record set `@id`s.

You can adjust the record set selection to focus on a specific table or collection.

In [ ]:
# Extract all records from each record set into a dictionary of DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '{record_set_id}' with columns: {df.columns.tolist()}")
        print(df.head(2), '\n')
    else:
        print(f"No records found for record set '{record_set_id}'.")

# Pick one record set with DataFrame for EDA below
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Use the first non-empty record set as example
    print(f"Main record set for further analysis: {main_record_set_id}")
    display_columns = dataframes[main_record_set_id].columns.tolist()
    print(f"Columns: {display_columns}")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No record set dataframes loaded for extraction.")

## 4. Exploratory Data Analysis (EDA)
Run basic filtering, normalization, and grouping for a numeric field using its `@id` as column name.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"\nFields in use (by @id):\n{df.columns.tolist()}")

    # Select a candidate numeric field by guessing (select first float/int column found)
    numeric_field_id = None
    for col in df.columns:
        # Try casting to numeric, skip if fails
        try:
            pd.to_numeric(df[col].dropna().iloc[0])
            numeric_field_id = col
            break
        except Exception:
            continue

    if numeric_field_id is not None:
        # Convert column to numeric (may coerce if string)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].median()  # for demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = numeric_field_id + "_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No main record set dataframe available for EDA.")

## 5. Visualization
Visualize the distribution of the main numeric field and groupings (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field was assigned
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we loaded the Croissant schema and metadata for the FAIR² dataset on adoption predictors of indigenous and modern knowledge in Kenyan rangelands. By referencing all entities via their `@id` fields, we extracted and previewed available record sets and performed basic exploratory analysis on available fields, including numeric normalization and group-based aggregation. Visualization steps illustrated key distributions for further exploration. For targeted research or analysis, please consult the full schema and data specification for additional fields and their descriptions.

**Note:** All columns and groups referenced in this notebook used their Croissant `@id` field to ensure robust provenance and schema tracing.